In [ ]:
#INTERNATIONAL TELECOMMUNICATION UNION - Indicators on Internet Usage & Broadband Connectivity

In [1]:
import pandas as pd

In [3]:
#loading in data
df = pd.read_csv("Indicator Data - Raw.csv")

In [4]:
#dataset included duplicate rows, this removes exact dupes
df = df.drop_duplicates()

In [5]:
#checking for duplicate country-year-indicator combinations
if df.duplicated(["AreaCode", "Year", "SeriesCode"]).any():
    raise ValueError("Multiple records found for the same country-year-indicator")

In [6]:
#pivoting data so each country-year has one row
ict_wide = df.pivot(
    index=["AreaCode", "AreaName", "Year"],
    columns="SeriesCode",
    values="Value"
).reset_index()

In [7]:
#renaming the columns of interest
indicator_names = {
    "i911": "mobile_cellular_per100",
    "i112": "fixed_telephone_subscriptions",
    "i96": "telecom_investment_pct_revenue",
    "i99H": "internet_users_pct",
    "xHH6_IDI": "households_internet_estimated_pct",
    "HHA422-%_HH--": "households_computer_pct",
    "HHU422-%_POP--": "computer_users_pct",
    "i93": "telecom_investment_pct_gdp"
}

ict_wide = ict_wide.rename(columns={
    "AreaCode": "country_code",
    "AreaName": "country",
    "Year": "year",
    **indicator_names
})

ict_wide.columns.name = None

In [8]:
#sort by country and year
ict_wide = ict_wide.sort_values(
    ["country_code", "year"]
).reset_index(drop=True)

#verifying there is only one row per country-year
assert not ict_wide.duplicated(["country_code", "year"]).any()

In [9]:
#saving cleaned dataset
ict_wide.to_csv("ict_country_year.csv", index=False)

In [10]:
#previewing
print(ict_wide.head())
print("Rows and columns:", ict_wide.shape)

  country_code country  year  households_computer_pct  computer_users_pct  \
0          ABW   Aruba  2000                      NaN                 NaN   
1          ABW   Aruba  2001                      NaN                 NaN   
2          ABW   Aruba  2002                      NaN                 NaN   
3          ABW   Aruba  2003                      NaN                 NaN   
4          ABW   Aruba  2004                      NaN                 NaN   

   fixed_telephone_subscriptions  mobile_cellular_per100  \
0                        38100.0               16.899314   
1                        37132.0               58.689995   
2                        37132.0               67.630418   
3                        37390.0               75.779439   
4                        38003.0              105.552874   

   telecom_investment_pct_gdp  telecom_investment_pct_revenue  \
0                         NaN                             NaN   
1                         NaN                 

In [11]:
#CODEBOOK
#creates one row per indicator with Series Code, Series, and Unit of Measure
codebook = (
    df[["SeriesCode", "Series", "Unit"]]
    .drop_duplicates()
    .rename(columns={
        "SeriesCode": "itu_code",
        "Series": "indicator_name",
        "Unit": "unit"
    })
)

#adding variable names
codebook["variable_name"] = codebook["itu_code"].map(indicator_names)

codebook = codebook[
    ["variable_name", "itu_code", "indicator_name", "unit"]
].sort_values("variable_name")

#calculates earliest and latest years with non-missing data
year_ranges = (
    df.dropna(subset=["Value"])
    .assign(Year=lambda x: pd.to_numeric(x["Year"], errors="raise"))
    .groupby("SeriesCode")["Year"]
    .agg(earliest_year="min", latest_year="max")
    .reset_index()
    .rename(columns={"SeriesCode": "itu_code"})
)

codebook = codebook.merge(
    year_ranges, on="itu_code", how="left", validate="one_to_one"
)

#adding the country, country code, and year variables
identifiers = pd.DataFrame({
    "variable_name": ["country_code", "country", "year"],
    "itu_code": ["", "", ""],
    "indicator_name": [
        "Country/economy code from ITU",
        "Country/economy name",
        "Year of the observation"
    ],
    "unit": ["Text", "Text", "Year"]
})

codebook = pd.concat([identifiers, codebook], ignore_index=True)

#calculates the percentage of nonmissing values in the cleaned dataset
pct_nonmissing = ict_wide.notna().mean().mul(100).round(2)

codebook["pct_nonmissing"] = (
    codebook["variable_name"].map(pct_nonmissing)
)

#reformats years to int
codebook[["earliest_year", "latest_year"]] = (
    codebook[["earliest_year", "latest_year"]].astype("Int64")
)

# Save and preview
codebook.to_csv("ict_codebook.csv", index=False)

print(codebook.to_string(index=False))

                    variable_name       itu_code                                                     indicator_name                           unit  earliest_year  latest_year  pct_nonmissing
                     country_code                                                     Country/economy code from ITU                           Text           <NA>         <NA>          100.00
                          country                                                              Country/economy name                           Text           <NA>         <NA>          100.00
                             year                                                           Year of the observation                           Year           <NA>         <NA>          100.00
               computer_users_pct HHU422-%_POP-- HH05 - Individuals who used a computer (from any location) - %_POP               % of individuals           2000         2025           15.25
    fixed_telephone_subscriptions           i